# Class 13 — Mastering Pandas Data Structures
## FinRoute Case Study: Full EDA + Relationship Mapping + 3-Phase Analytics

**Nine parts building on each other:**
1. Series — creation, indexing, alignment, NaN
2. DataFrame — inspection checklist, loc/iloc, column ops
3. Boolean indexing — single, multi-condition, `.query()`, `.isin()`, `.str.contains()`
4. Relationship analysis — cardinality mapping (1:1, 1:Many, M:M)
5. EDA workflow — quality checks, distributions, visualisation
6. Phase 1 — separate E05 (recoverable) from E01 (not recoverable)
7. Phase 2 — break-even model, algebraic solution + DataFrame simulation
8. Phase 3 — chargeback risk via merge + groupby + multi-agg
9. What-if simulation — optimal routing impact

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
print(f'Pandas {pd.__version__} | NumPy {np.__version__}')

Pandas 2.2.2 | NumPy 2.0.2


---
## Part 1: Series
### 1.1 Three creation patterns

In [ ]:
# From list + custom index
sales = pd.Series([250, 300, 150, 400], index=['Mon','Tue','Wed','Thu'])
print(sales)
print(f'\nType: {type(sales)} | dtype: {sales.dtype}')

Mon    250
Tue    300
Wed    150
Thu    400
dtype: int64

Type: <class 'pandas.core.series.Series'> | dtype: int64


In [ ]:
# From dict — keys become the index automatically
ages = pd.Series({'Alice': 28, 'Bob': 34, 'Charlie': 25})
print(ages)

Alice      28
Bob        34
Charlie    25
dtype: int64


### 1.2 Accessing and filtering

In [ ]:
# Three access patterns
print('Label access  ["Wed"]:', sales['Wed'])
print('Position      iloc[0]:', sales.iloc[0])
print('Last item     iloc[-1]:', sales.iloc[-1])

# Vectorised filter — no loop needed
high = sales[sales > 250]
print('\nHigh sales (>250):')
print(high)

# Vectorised arithmetic
print('\nAfter 10% growth:')
print((sales * 1.10).round(0))

Label access  ["Wed"]: 150
Position      iloc[0]: 250
Last item     iloc[-1]: 400

High sales (>250):
Tue    300
Thu    400
dtype: int64

After 10% growth:
Mon    275.0
Tue    330.0
Wed    165.0
Thu    440.0
dtype: float64


### 1.3 Index alignment and NaN

In [ ]:
s1 = pd.Series([10, 20, 30], index=['A','B','C'])
s2 = pd.Series([5,  10, 15], index=['B','C','D'])

print('s1 + s2 (aligned by label — NaN where no match):')
print(s1 + s2)

print('\nWith fill_value=0 (no NaN):')
print(s1.add(s2, fill_value=0))

print('\ndropna():', (s1+s2).dropna().to_dict())

s1 + s2 (aligned by label — NaN where no match):
A     NaN
B    25.0
C    40.0
D     NaN
dtype: float64

With fill_value=0 (no NaN):
A    10.0
B    25.0
C    40.0
D    15.0
dtype: float64

dropna(): {'B': 25.0, 'C': 40.0}


---
## Part 2: DataFrame
### 2.1 Creation and inspection

In [ ]:
df = pd.DataFrame({
    'product_id': ['P01','P02','P03','P04'],
    'category':   ['Electronics','Office','Electronics','Furniture'],
    'price':      [899.99, 24.50, 150.00, 320.75],
    'in_stock':   [True, True, False, True]
})
df

,product_id,category,price,in_stock
0,P01,Electronics,899.99,True
1,P02,Office,24.50,True
2,P03,Electronics,150.00,False
3,P04,Furniture,320.75,True


In [ ]:
# THE INSPECTION CHECKLIST — run these on every new dataset
print(f'Shape: {df.shape}')
print(f'\nNull values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().any()}')
print(f'\nInfo:')
df.info()

Shape: (4, 4)

Null values:
product_id    0
category      0
price         0
in_stock      0
dtype: int64

Duplicates: False

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  4 non-null      object 
 1   category    4 non-null      object 
 2   price       4 non-null      float64
 3   in_stock    4 non-null      bool   
dtypes: bool(1), float64(1), object(2)
memory usage: 232.0+ bytes


In [ ]:
print('Describe:')
df.describe()

Describe:


,price
count,4.000000
mean,348.810000
std,386.992312
min,24.500000
25%,118.625000
50%,235.375000
75%,465.560000
max,899.990000


### 2.2 loc vs iloc

In [ ]:
df_idx = df.set_index('product_id')
print('loc["P03"] — by label:')
print(df_idx.loc['P03'])
print()
print('loc[["P01","P04"], ["category","price"]] — multi-label:')
print(df_idx.loc[['P01','P04'], ['category','price']])
print()
print('iloc[0:2] — by position:')
print(df_idx.iloc[0:2])

loc["P03"] — by label:
category    Electronics
price             150.0
in_stock          False
Name: P03, dtype: object

loc[["P01","P04"], ["category","price"]] — multi-label:
               category   price
product_id                     
P01         Electronics  899.99
P04           Furniture  320.75

iloc[0:2] — by position:
               category   price  in_stock
product_id                               
P01         Electronics  899.99      True
P02              Office   24.50      True


### 2.3 Column manipulation

In [ ]:
df['discounted_price'] = df['price'] * 0.90             # add
df = df.drop(columns=['in_stock'])                       # remove
df['price_tier'] = df['price'].apply(                    # apply fn
    lambda x: 'Premium' if x > 300 else 'Standard'
)
df

,product_id,category,price,discounted_price,price_tier
0,P01,Electronics,899.99,809.991,Premium
1,P02,Office,24.50,22.050,Standard
2,P03,Electronics,150.00,135.000,Standard
3,P04,Furniture,320.75,288.675,Premium


---
## Part 3: Boolean Indexing

In [ ]:
np.random.seed(42)
sales_data = pd.DataFrame({
    'date':       pd.date_range('2024-01-01', periods=10),
    'store_id':   np.random.choice([101,102,103],10),
    'revenue':    np.random.randint(1000,5000,10),
    'units_sold': np.random.randint(10,100,10)
})
sales_data

,date,store_id,revenue,units_sold
0,2024-01-01,103,1330,11
1,2024-01-02,101,2482,97
2,2024-01-03,103,3135,39
3,2024-01-04,103,4444,47
4,2024-01-05,101,4171,11
5,2024-01-06,101,3919,73
6,2024-01-07,103,4735,69
7,2024-01-08,102,1130,30
8,2024-01-09,103,2685,42
9,2024-01-10,103,4380,85


In [ ]:
# Single condition
high = sales_data[sales_data['revenue'] > 3000]
print(f'Revenue > 3000: {len(high)} rows')
high

Revenue > 3000: 6 rows


,date,store_id,revenue,units_sold
2,2024-01-03,103,3135,39
3,2024-01-04,103,4444,47
4,2024-01-05,101,4171,11
5,2024-01-06,101,3919,73
6,2024-01-07,103,4735,69
9,2024-01-10,103,4380,85


In [ ]:
# Multiple conditions — ALWAYS wrap each in ()
complex_f = sales_data[
    (sales_data['store_id'] == 101) & (sales_data['units_sold'] > 50)
]
print(f'Store 101 AND units > 50: {len(complex_f)} rows')
complex_f

Store 101 AND units > 50: 2 rows


,date,store_id,revenue,units_sold
1,2024-01-02,101,2482,97
5,2024-01-06,101,3919,73


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=complex_f)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# .query() — SQL-like string syntax (cleaner for 3+ conditions)
result = sales_data.query('revenue > 3000 and units_sold > 40')
print(f'.query() result: {len(result)} rows')
result

.query() result: 4 rows


,date,store_id,revenue,units_sold
3,2024-01-04,103,4444,47
5,2024-01-06,101,3919,73
6,2024-01-07,103,4735,69
9,2024-01-10,103,4380,85


In [ ]:
# .isin() — SQL IN clause
active = sales_data[sales_data['store_id'].isin([101, 103])]
print(f'.isin([101,103]): {len(active)} rows')

.isin([101,103]): 9 rows


In [ ]:
# .str.contains() — substring filtering on text columns
df_m = pd.DataFrame({'merchant': ['Swiggy_Food','Zomato_Food','Netflix_Ent','BookMyShow_Ent']})
food = df_m[df_m['merchant'].str.contains('Food')]
print('Food merchants:', food['merchant'].tolist())

Food merchants: ['Swiggy_Food', 'Zomato_Food']


---
## Generate FinRoute Data Lake

In [ ]:
n = 10000
df_txns = pd.DataFrame({
    'txn_id': [f'TXN_{i}' for i in range(1,n+1)],
    'merchant_id': np.random.choice([f'M_{i}' for i in range(100,150)], n),
    'amount_usd': np.round(np.random.exponential(scale=150, size=n), 2),
    'routed_acquirer': np.random.choice(['Bank A','Bank B'], n, p=[0.8,0.2]),
})
def assign_status(row):
    if row['routed_acquirer']=='Bank A':
        return np.random.choice(['Success','Failed_E05_Timeout','Failed_E01_NSF'],p=[0.85,0.10,0.05])
    return np.random.choice(['Success','Failed_E01_NSF'],p=[0.98,0.02])
df_txns['status'] = df_txns.apply(assign_status, axis=1)
merchants = df_txns['merchant_id'].unique()
df_merchants = pd.DataFrame({'merchant_id':merchants,'mcc_code':np.random.choice(['Retail','Digital Goods','Travel','Crypto'],len(merchants)),'vintage_months':np.random.randint(1,60,len(merchants)),'chargeback_rate':np.random.uniform(0.001,0.05,len(merchants))})
df_costs = pd.DataFrame({'acquirer_bank':['Bank A','Bank B'],'fixed_cost_usd':[0.10,0.25],'variable_cost_pct':[0.005,0.012]})
print(f'df_txns: {df_txns.shape} | df_merchants: {df_merchants.shape} | df_costs: {df_costs.shape}')

df_txns: (10000, 5) | df_merchants: (50, 4) | df_costs: (2, 3)


---
## Part 4: Relationship Analysis
Before any analysis — map how your tables relate. This prevents double-counting and wrong joins.

In [ ]:
# TxnID → MerchantID: One-to-One
is_u = df_txns['txn_id'].is_unique
pairs = df_txns[['txn_id','merchant_id']].drop_duplicates().shape[0]
print(f'txn_id unique: {is_u} | Unique pairs: {pairs} = {len(df_txns)} rows → 1:1')

# MerchantID → TxnID: One-to-Many
m_counts = df_txns.groupby('merchant_id')['txn_id'].nunique()
print(f'\nMax txns per merchant: {m_counts.max()} → 1:Many')
print(m_counts.nlargest(5))

# Acquirer → TxnID: One-to-Many
acq = df_txns.groupby('routed_acquirer')['txn_id'].nunique()
print(f'\nAcquirer → TxnID:\n{acq}')

txn_id unique: True | Unique pairs: 10000 = 10000 rows → 1:1

Max txns per merchant: 232 → 1:Many
merchant_id
M_125    232
M_144    225
M_148    225
M_122    223
M_127    222
Name: txn_id, dtype: int64

Acquirer → TxnID:
routed_acquirer
Bank A    7998
Bank B    2002
Name: txn_id, dtype: int64


---
## Part 5: EDA

In [ ]:
# Status distribution with percentage
print('STATUS DISTRIBUTION:')
print(df_txns['status'].value_counts())
print()
print(df_txns['status'].value_counts(normalize=True).mul(100).round(2))

STATUS DISTRIBUTION:
status
Success               8813
Failed_E05_Timeout     756
Failed_E01_NSF         431
Name: count, dtype: int64

status
Success               88.13
Failed_E05_Timeout     7.56
Failed_E01_NSF         4.31
Name: proportion, dtype: float64


In [ ]:
# Amount distribution stats
print('AMOUNT_USD STATISTICS:')
print(df_txns['amount_usd'].describe().round(2))
print(f'Median: ${df_txns["amount_usd"].median():.2f}')

AMOUNT_USD STATISTICS:
count    10000.00
mean       150.05
std        149.15
min          0.02
25%         44.86
50%        104.70
75%        206.18
max       1370.94
Name: amount_usd, dtype: float64
Median: $104.70


---
## Phase 1: Quantify Lost Revenue
**Critical distinction:** E05 = Bank A's server error (recoverable). E01 = User's bank declined (NOT recoverable).

In [ ]:
df_timeouts = df_txns[df_txns['status'] == 'Failed_E05_Timeout']
df_nsf      = df_txns[df_txns['status'] == 'Failed_E01_NSF']

lost_mdr_e05 = df_timeouts['amount_usd'].sum() * 0.02
lost_mdr_e01 = df_nsf['amount_usd'].sum() * 0.02

print(f'E05 Timeout: {len(df_timeouts):,} txns | MDR lost: ${lost_mdr_e05:,.2f}/day | Ann: ${lost_mdr_e05*365:,.0f}')
print(f'E01 NSF:     {len(df_nsf):,} txns | MDR lost: ${lost_mdr_e01:,.2f}/day | Ann: ${lost_mdr_e01*365:,.0f}')
print()
print('E05 = fixable by routing infrastructure. E01 = not our problem.')

E05 Timeout: 756 txns | MDR lost: $2,255.89/day | Ann: $823,401
E01 NSF:     431 txns | MDR lost: $1,353.02/day | Ann: $493,851

E05 = fixable by routing infrastructure. E01 = not our problem.


---
## Phase 2: Break-Even Analysis

In [ ]:
# Algebraic solution: when does Profit_A = Profit_B?
# Profit_A = 0.02x - 0.10 - 0.005x = 0.015x - 0.10
# Profit_B = 0.02x - 0.25 - 0.012x = 0.008x - 0.25
# Set equal: 0.015x - 0.10 = 0.008x - 0.25
# 0.007x = 0.15  →  x = 21.43
breakeven = 0.15 / 0.007
print(f'Exact break-even: $0.15 / 0.007 = ${breakeven:.2f}')

Exact break-even: $0.15 / 0.007 = $21.43


In [ ]:
# Build the DataFrame model
test = pd.DataFrame({'txn_amount': [5, 10, 20, 21.43, 22, 50, 100, 200, 500]})
MDR = 0.02
test['profit_A'] = test['txn_amount']*MDR - 0.10 - test['txn_amount']*0.005
test['profit_B'] = test['txn_amount']*MDR - 0.25 - test['txn_amount']*0.012
test['route_to'] = np.where(test['profit_B'] >= test['profit_A'], 'Bank B', 'Bank A')
test.round(4)

,txn_amount,profit_A,profit_B,route_to
0,5.00,-0.0250,-0.2100,Bank A
1,10.00,0.0500,-0.1700,Bank A
2,20.00,0.2000,-0.0900,Bank A
3,21.43,0.2214,-0.0786,Bank A
4,22.00,0.2300,-0.0740,Bank A
5,50.00,0.6500,0.1500,Bank A
6,100.00,1.4000,0.5500,Bank A
7,200.00,2.9000,1.3500,Bank A
8,500.00,7.4000,3.7500,Bank A


---
## Phase 3: Risk Analytics — Merge + GroupBy

In [ ]:
# LEFT JOIN: all transactions + merchant data where available
df_enriched = pd.merge(df_txns, df_merchants, how='left', on='merchant_id')
df_success = df_enriched[df_enriched['status']=='Success'].copy()
df_success['expected_cb_loss'] = (df_success['amount_usd'] + 20) * df_success['chargeback_rate']

risk_by_mcc = df_success.groupby('mcc_code').agg(
    total_volume   = ('amount_usd',       'sum'),
    txn_count      = ('txn_id',           'count'),
    avg_cb_rate    = ('chargeback_rate',  'mean'),
    estimated_loss = ('expected_cb_loss', 'sum'),
    max_cb_rate    = ('chargeback_rate',  'max'),
).reset_index().sort_values('estimated_loss', ascending=False)
risk_by_mcc.round(2)

,mcc_code,total_volume,txn_count,avg_cb_rate,estimated_loss,max_cb_rate
1,Digital Goods,433179.35,2861,0.03,15001.96,0.05
3,Travel,447563.91,3022,0.02,11685.65,0.05
0,Crypto,313415.83,2067,0.03,9103.62,0.05
2,Retail,125941.85,863,0.03,3586.42,0.05


---
## Part 9: What-If Simulation

In [ ]:
bank_a_sr = (df_txns[df_txns['routed_acquirer']=='Bank A']['status']=='Success').mean()
bank_b_sr = (df_txns[df_txns['routed_acquirer']=='Bank B']['status']=='Success').mean()
print(f'Bank A success rate: {bank_a_sr*100:.1f}%')
print(f'Bank B success rate: {bank_b_sr*100:.1f}%')

candidates = df_timeouts[df_timeouts['amount_usd'] >= breakeven]
potential = candidates['amount_usd'].sum() * (bank_b_sr - bank_a_sr) * MDR
print(f'\nReroutable E05 transactions (>${breakeven:.0f}): {len(candidates):,}')
print(f'Estimated daily MDR recovery: ${potential:,.2f}')
print(f'Annualised recovery: ${potential*365:,.0f}')

Bank A success rate: 85.6%
Bank B success rate: 98.3%

Reroutable E05 transactions (>$21): 656
Estimated daily MDR recovery: $282.62
Annualised recovery: $103,158
